In [21]:
%pip install chromadb 
%pip install google-generativeai

%pip install faiss-cpu 


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install torch sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os 


In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Siddhanth\AppData\Local\Temp\ipykernel_21948\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\Siddhanth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] ='pdf'
            
            all_documents.extend(documents)
            print(f"loaded{len(documents)} pages")
        except:
            print(f"Error:{e}")
    print(f"total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")
            

Found 3 PDF files to process

Processing: Axxela.pdf


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


loaded1 pages

Processing: UBS.pdf
loaded3 pages

Processing: ZS_Case_Frameworks.pdf
loaded16 pages
total documents loaded: 20


In [8]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """spit documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap,
        length_function = len, 
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents  into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)

chunks

Split 20 documents  into 31 chunks

Example chunk:
Content: About Us: 
We are a firm with a strong presence in the Futures and Derivatives related operations in the 
International Capital Markets. We have built ourselves a reputation for grooming university 
g
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-03T13:17:45+00:00', 'author': 'Megha Gupta', 'moddate': '2026-06-03T13:17:46+00:00', 'source': '..\\data\\pdf\\Axxela.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Axxela.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-03T13:17:45+00:00', 'author': 'Megha Gupta', 'moddate': '2026-06-03T13:17:46+00:00', 'source': '..\\data\\pdf\\Axxela.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Axxela.pdf', 'file_type': 'pdf'}, page_content='About Us: \nWe are a firm with a strong presence in the Futures and Derivatives related operations in the \nInternational Capital Markets. We have built ourselves a reputation for grooming university \ngraduates, aspiring to leverage their analytical skills and emotional intelligence, into quick \nthinker’s adept at handling competitive situations professionally. \nWith this email we want to offer immense growth opportunities to your students. For further \ndetails: - https://www.axxela.in/ \n \nRole Details: \n⮚ To be involved in the execution of trades in commodities and fixed income in International \nMarkets. \n⮚ Utilize strong risk managem

In [10]:


import numpy as np 
from sentence_transformers import SentenceTransformer 
import chromadb 
from chromadb.config import Settings 
import uuid 
from typing  import List, Dict, Any, Tuple 
from sklearn.metrics.pairwise  import cosine_similarity

In [12]:
class EmbeddingManager:
    """Handles document embeddings  generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialise the embedding manager
        
        Args: 
            model-name: HuggingFace  model name for sentence embeddings """
        
        self.model_name = model_name 
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load  the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
        #     print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
            
        Returns: 
            numpy array of embeddings with shape {len(texts), embeddings_dim}
            
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts....")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

   
    
 

In [13]:
 #initilise the embeddings manger 
embedding_manager  = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2361.92it/s]


In [14]:
class VectorStore:
    """Manages document embeddings ina ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persistent_directory:  str = "../data/vector_store"):
        """Initialise the vector store
        
        Args:
        collection_name : Name of the ChromaDB collection 
        persistent_directory: Directory to persist the vector store """

        self.collection_name = collection_name
        self.persist_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialise_store()
    
    def _initialise_store(self):
        """Initialise ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path =self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description" : "PDF doucment embeddings for RAG"}
            )

            print(f"Vector store initiliased. Collection: {self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error initialisig vector store: {e}")
            raise   

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store
        
        Args:
        documents: List of Langchain documents
        embeddings: Correspoding embeddings for the documents"""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match numbe rof embeddings")
    
        print(f"Adding {len(documents)} documents to vector store.....")

        ids = []
        metadatas =[]
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique ID
            doc_id  = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i 
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list, 
                metadatas = metadatas, 
                documents = documents_text
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            

        except Exception as e:
            print(f"Error adding docuements to vector store: {e}")
    

In [15]:
vectorstore = VectorStore()
vectorstore

Vector store initiliased. Collection: pdf_documents
Existing documents in collection:0


In [17]:
texts = [doc.page_content[:200] for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 31 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.86it/s]

Generated embeddings with shape: (31, 384)
Adding 31 documents to vector store.....
Sucessfully added 31 documents to vector store
Total documents in collection: 31


In [26]:
class RAGRetriever:
    """Handles retrieving based on query from vector store """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Iniitlise  the retriever
        Args:
        Vectorstore: Vector Store  containing document embeddings
        embedding_manager: Manager for generating the query embeddings
        
        """

        self.vector_store = vector_store 
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5,  score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant docuemnts for a query
        
        Args:
            query: The serach query 
            Top_k: Number of top results to return 
            score_financial: Minimum similarity score threshold 

        Returns:
            list of dictionaries containing retrieved docuements and metdata
              
             """

        print(f"Retrieving docuemnts for query: {query}")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
    
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )
        
            retrieved_docs =[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas =results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similiarity_score = 1 - distance

                    if  similiarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content': document, 
                                'metadata' : metadata,
                                'similarity_score' :  similiarity_score,
                                'distance' : distance,
                                'rank' : i + 1
                            }
                        )
                        print(f"Retrieverd{len(retrieved_docs)} documents(after fitlering)")
                    
            else:
                print("No documents found")
            
            return retrieved_docs 


        except Exception as e:
            print(f"{e}")
            return[]
        
retriever = RAGRetriever(vectorstore, embedding_manager)

retriever

In [27]:
retriever.retrieve("What ZS Framwork?")

Retrieving docuemnts for query: What ZS Framwork?
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.02it/s]

Generated embeddings with shape: (1, 384)
Retrieverd1 documents(after fitlering)


[{'id': 'doc_cd29af24_8',
  'content': 'ZS grades how you think, not the final number. Structure, narrating your logic, sanity-checking, \nand adapting when nudged matter more than landing the exact figure. \n \nThe Universal Spine (use on every single case) \n1. Clarify & restate the objective. \nRepeat the goal back and pin the success metric. Ask 2–3 clarifying questions before structuring. \nAlways nail down: \n• What\'s the metric? (profit / market share / revenue / votes / cost saved) \n• What\'s the timeframe / constraint? \n• What does "success" look like to the client? \n \n2. Structure out loud, then get buy-in. \nState your framework as 3–4 MECE buckets. \nEnd with: \n"Does it make sense if I start here?"',
  'metadata': {'file_type': 'pdf',
   'producer': 'Microsoft® Word 2021',
   'creator': 'Microsoft® Word 2021',
   'author': 'Sarthak Mane',
   'source_file': 'ZS_Case_Frameworks.pdf',
   'page': 0,
   'doc_index': 8,
   'source': '..\\data\\pdf\\ZS_Case_Frameworks.pdf',


In [23]:
import google.generativeai as genai
import os 
import dotenv
from dotenv import load_dotenv

load_dotenv()

api = os.getenv("GEMINI_API_KEY")

# Configure Gemini API
genai.configure(api_key=api)

# Create model
llm = genai.GenerativeModel(
    model_name="gemini-2.5-flash"
)

In [28]:
def rag_simple(query, retriever, top_k=3):
    # Retrieve relevant chunks
    results = retriever.retrieve(query, top_k=top_k)

    # Combine retrieved chunks
    context = "\n\n".join(
        [doc["content"] for doc in results]
    ) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Prompt
    prompt = f"""
    Use the following context to answer the question concisely.

    Context:
{context}

Question: {query}

Answer:
"""

    # Generate response using Gemini
    response = llm.generate_content(prompt)

    return response.text

In [30]:
query = "What is ZS case framework?"

answer = rag_simple(
    query=query,
    retriever=retriever,
    top_k=3
)

print(answer)

Retrieving docuemnts for query: What is ZS case framework?
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.01it/s]

Generated embeddings with shape: (1, 384)
Retrieverd1 documents(after fitlering)


A ZS case framework is an opening framework or structure that a candidate proposes to begin a ZS case, providing initial organization before moving into data analysis and synthesis.
